In [ ]:
from dataclasses import dataclass

type Id = int
type Path = tuple[str, ...] # but maybe (Id,str,Id) would be appropriate? Then we can check for correctness, derive codomain etc.
type FatId = tuple[Path, Id]
@dataclass
class UF:
    parents : list[FatId]
    loops : list[set[Path]] # "Lattice" (set) of loops
    def find(self, x: FatId) -> FatId:
        pf, id = x
        while True:
            p, id2 = self.parents[id]
            if id == id2:
                return (pf, id)
            pf = pf + p
            id = id2
    def makeset(self):
        id = len(self.parents)
        self.parents.append(((), id))
        return ((),), id
    def union(self, x: FatId, y: FatId): # why/explain parameter?
        x,y = self.find(x), self.find(y)
        xpf, xid = x
        ypf, yid = y
        if xid != yid:
            self.parents[xid] = (xpf + reversed(ypf), yid)
        else:
            self.loops.append({xpf + reversed(ypf)})
def move(p : Path, x : FatId):
    return (p + x[0], x[1])


Combine string KB at proof level with union find at bottom level.

https://smimram.github.io/ocaml-alg/

squier completion
https://webusers.imj-prg.fr/~yves.guiraud/articles/polybook.pdf

UF isn't collapsing proofs really. We could do that. Proof irrelevance I guess.
Compare with a more ordinary proof producing union find

FatID with proof kind of reminds me of dirac belt trick or jordan wigner string or anyon paths. We have the thing here, yes, but there is a worldline saying where it came from.


In [ ]:
class UF2:
    string_rewrites : list[tuple[Path, Path]]
    parents : list[FatId]
    loops : list[set[Path]]

    def rebuild(self):
        # string KB
        # normalize loops according to the string rewrites
        # also loops shoul prune anything that is just a multiple. loop_generators
        # we could have done this pruning in UF1 also

The group union find is not

You can kind of make a null groupoid out of a group and a set of objects.
(ob, grp) pairs
and there are morphisms (dom, grp, cod)

Prearrows / Arr
def infer_cod(Ob, Arr) -> Option(Ob) # cod
def infer_dom(Ob, Arr) -> 

infer_cod(a, id) = a

Inferarrow,  object, prearrow -> we can infer cod

id(a)

linear maps - pullbacks

pullback is unique up to isomorphisms. isomorphisms will abound in this thing.
https://en.wikipedia.org/wiki/Pullback_(category_theory)#Least_common_multiple least common miltiple is pullback

A groupoid with just copies of the group on every edge.
linear transformations. Again dots are different 1-d vector spaces. Copies

- ->



In [ ]:
type Arr = object
type Obj = int

type FatId = tuple[Arr, Obj]

class PBArr(Protocol):
    def pb(f, g) -> (Option Self, Option Self) # allow for the case where pullback is equalizer / can reuse a/b
        # forall a b c f g, hom(f,a,c), hom(g,b,c) -> exists h d, pb1,pb2, f. pb1 = g. pb2  and d best 
        # The exists d is skolemized by sometimesn eeding makeset

class PullbackUF:
    def union(self, a : FatId, b : FatId) -> None:
        # there is an implicit c object a -annota> c <annotb- b  
        # infer_cod(a : FatId) == infer_cod(b : FatId)
        # 

Finset thinnings. Thinnings + permutations / injective finmaps
Actually, yeah. Specializing the relation of alpha pemruation doesn't seem that important compared to scope? Alpha permuted things aren't equal unless the theory says they are.

Noninjective thinmaps also let us notice narrowing   f(X,Y) narrows f(X,X) . A curious kind of equality assumption kind of. 
lam xy , f(X,Y) = g(X,Y)  implies lam x, f(X,X) == g(X,X) . This is true in interpretations I've been considering, but has not been specialized out

thinnings   f(t(X), s(Y)). Lifting that out would make every variable unique in an expressoin with equality joining at the top. Maybe not feasible (?) because expressions oculd have inf vars?

subst(t, x, y)

Substitution pounded in there.


In [13]:
from dataclasses import dataclass
class Term:
    pass
@dataclass
class App(Term):  # Comp(f, args)  but specialize like Cons compared to append. ConsComp
    f : str
    args : MultiTerm
    def shift(self, d : int) -> App:
        return App(f=self.f, args=self.args.shift(d))
    def substitute(self, t : MultiTerm) -> App:
        return App(f=self.f, args=self.args.substitute(t))

@dataclass
class Var(Term):
    n : int
    def shift(self, d : int) -> Var:
        return Var(n=self.n + d)
    def substitute(self, t : MultiTerm) -> Var:
        return t.ts[self.n]
@dataclass
class MultiTerm:
    d :  int
    ts : list[Term]
    def __or__(self, other : MultiTerm) -> MultiTerm: # truly independet paralle compose
        return MultiTerm(d=self.d + other.d, ts=self.ts + other.shift(self.d).ts)
    def __mul__(self, other : MultiTerm) -> MultiTerm: # 
        assert self.d == other.d
        return MultiTerm(d=self.d, ts=self.ts + other.ts)
    def __matmul__(self, other : MultiTerm) -> MultiTerm: # sequential compose
        return self.substitute(other)
    def substitute(self, t : MultiTerm) -> MultiTerm:
        assert self.d == len(t.ts)
        return MultiTerm(d=self.d, ts=[s.substitute(t) for s in self.ts])
    def shift(self, d : int) -> MultiTerm:
        return MultiTerm(d=self.d + d, ts=[t.shift(d) for t in self.ts])
    def weaken(self, n : int) -> MultiTerm: # I guess we could weaken using a thinning.
        return MultiTerm(d=self.d + n, ts=self.ts)
    


def app(f: str, *args : MultiTerm) -> MultiTerm:
    d = args[0].d
    assert all(a.d == d for a in args) and all(len(a.ts) == 1 for a in args)
    return MultiTerm(d=d, ts=[App(f=f, args=MultiTerm(d=args[0].d, ts=list(args)))])
def const(f, d : int) -> MultiTerm:
    return MultiTerm(d=d, ts=[App(f=f, args=MultiTerm(d=d, ts=[]))])

def var(n : int, d : int) -> MultiTerm:
    return MultiTerm(d=d, ts=[Var(n)])

id_ = var(0, 1)

v0 = var(0, 1)
app("f", v0).substitute(app("g", v0))

swap = MultiTerm(d=2, ts=[var(1,2),var(0,2)])
swap
f = app("f", var(0,1)) # f(X) as representing f
g = app("g", var(0,2), var(1,2)) # g(X,Y) as representing g

fst = var(0,2)
snd = var(1,2)
proj = var

from pprint import pprint
pprint(g @ ((f @ f) * v0))


MultiTerm(d=2,
          ts=[App(f='g',
                  args=MultiTerm(d=2,
                                 ts=[MultiTerm(d=2,
                                               ts=[App(f='f',
                                                       args=MultiTerm(d=1,
                                                                      ts=[MultiTerm(d=1,
                                                                                    ts=[App(f='f',
                                                                                            args=MultiTerm(d=1,
                                                                                                           ts=[MultiTerm(d=1,
                                                                                                                         ts=[Var(n=0)])]))])]))]),
                                     MultiTerm(d=2, ts=[Var(n=0)])]))])


In [ ]:
from dataclasses import dataclass

class Cat: ...

@dataclass
class Comp(Cat): 
    f : Cat
    g : Cat
    # or could flatten associativity?

@dataclass
class Id(Cat): # var
    pass

@dataclass
class Decl(Cat):
    f : str
    arity : int

@dataclass
class Fork(Cat):
    # same domain. but product over. Hmm. That's just the produce?
    f : Cat
    g : Cat




SyntaxError: invalid syntax (1677859489.py, line 21)

  t1 <- vs  > t1
product diagram

exists vs -> t12  with projections.   Yes
* is literally the categorical product. (but without giving the projects)

 

Multiterm flavored egraph

type FatId = (thin, SmallVec[rawId])
type FatId = Vec[ThinId]   yes.
Node = 
| {f : String, FatId}
| Var


FatId.prod()


rgsvd



